# 10-714 Homework 4

In this homework, you will leverage all of the components built in the last three homeworks to solve some modern problems with high performing network structures. We will start by adding a few new ops leveraging our new CPU/CUDA backends. Then, you will implement convolution, and a convolutional neural network to train a classifier on the CIFAR-10 image classification dataset. Then, you will implement recurrent and long-short term memory (LSTM) neural networks, and do word-level prediction language modeling on the Penn Treebank dataset.

As always, we will start by copying this notebook and getting the starting code.
Reminder: __you must save a copy in drive__.

In [1]:
# Mount Drive (same as before)
from google.colab import drive
drive.mount('/content/drive')

# %cd /content/drive/MyDrive/10714

# Clone YOUR repo into a folder named "hw4"
# !git clone https://github.com/3N3G/dlsys-final.git proj
%cd /content/drive/MyDrive/10714/proj

# Same installs as before
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

Mounted at /content/drive
/content/drive/MyDrive/10714/proj
  Cloning https://github.com/dlsys10714/mugrade.git to /tmp/pip-req-build-gsfi47y1
  Running command git clone --filter=blob:none --quiet https://github.com/dlsys10714/mugrade.git /tmp/pip-req-build-gsfi47y1
  Resolved https://github.com/dlsys10714/mugrade.git to commit ac73f725eb2ce0e2c6a38fa540035ee970b8b873
  Preparing metadata (setup.py) ... done
  Created wheel for mugrade: filename=mugrade-1.3-py3-none-any.whl size=3708 sha256=ed62a7665cf06d860d136a9ad261ff3190c165bcb88d8cc181a013aa2a68b3c6
  Stored in directory: /tmp/pip-ephem-wheel-cache-ksg5zjzr/wheels/df/c7/14/2b747145fc762900af3ff05bd0c9192c506e70db3ef3890239
Successfully built mugrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 6.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

# 1. Load credentials from Secrets
token = userdata.get('GITHUB_TOKEN')
email = userdata.get('GIT_EMAIL')
name = userdata.get('GIT_NAME')
repo_url = "github.com/3N3G/dlsys-final.git" # Your specific repo
repo_name = "dlsys-final"

# 2. Configure Git (Global)
!git config --global user.email "$email"
!git config --global user.name "$name"
!git remote set-url origin https://$token@$repo_url

In [ ]:
!make

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Found pybind11: /usr/local/lib/python3.12/dist-packages/pybind11/include (found version "3.0.1")
-- Found cuda, building cuda backend
-- Configuring done (0.7s)
-- Generating done (0.5s)
-- Build files have been written to: /content/drive/MyDrive/10714/proj/build
make[1]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[2]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[3]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[3]: Leaving directory '/content/drive/MyDrive/10714/proj/build'
[  0%] Built target ndarray_backend_cpu
make[3]: Entering directory '/content/drive/

In [2]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

env: PYTHONPATH=./python
env: NEEDLE_BACKEND=nd


In [3]:
import sys
sys.path.append('./python')

In [ ]:
# Download the datasets you will be using for this assignment

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

# Download CIFAR-10 dataset
if not os.path.isdir("./data/cifar-10-batches-py"):
    urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", "./data/cifar-10-python.tar.gz")
    !tar -xvzf './data/cifar-10-python.tar.gz' -C './data'

You will now use your convolutional layer to implement a model similar to _ResNet9_, which is known to be a reasonable model for getting good accuracy on CIFAR-10 quickly (see [here](https://github.com/davidcpage/cifar10-fast)). Our main change is that we used striding instead of pooling and divided all of the channels by 4 for the sake of performance (as our framework is not as well-optimized as industry-grade frameworks).

In the figure below, before the first linear layer, you should "flatten" the tensor. You can use the module `Flatten` in `nn_basic.py`, or you can simply use `.reshape` in the `forward()` method of your ResNet9.

Make sure that you pass the device to all modules in your model; otherwise, you will get errors about mismatched devices when trying to run with CUDA.

<center><img src="https://github.com/dlsyscourse/hw4/blob/main/ResNet9.png?raw=true" alt="ResNet9" style="width: 400px;" /></center>

We have tried to make it easier to pass the tests here than for previous assignments where you have implemented models. In particular, we are just going to make sure it has the right number of parameters and similar accuracy and loss after 1 or 2 batches of CIFAR-10.

Now, you can train your model on CIFAR-10 using the following code. Note that this is likely going to be quite slow, and also  not all that accurate due to the lack of data augmentation. You should expect it to take around 500s per epoch.

In [ ]:
import sys
sys.path.append('./python')
sys.path.append('./apps')

import importlib
import needle
import models
import simple_ml
importlib.reload(needle.ops.ops_mathematic)
importlib.reload(needle.ops)
importlib.reload(needle.nn.nn_basic)  # if you added MaxPool2d there
importlib.reload(needle.nn)
importlib.reload(needle)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9, CifarNetNeedle
from simple_ml import train_cifar10, evaluate_cifar10

print("HERE")
device = ndl.cuda()
print("Using device:", device)
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
print("LOADING DATA")
dataloader = ndl.data.DataLoader(\
         dataset=dataset,
         batch_size=32,
         shuffle=True,)
print("MODEL")
# model = ResNet9(device=device, dtype="float32")
model = CifarNetNeedle(device=device, dtype="float32")
print("TRAINING")
train_cifar10(model, dataloader, n_epochs=30, optimizer=ndl.optim.Adam,
      lr=0.001, weight_decay=0.001)
evaluate_cifar10(model, dataloader)

Using needle backend
HERE
Using device: cuda()
LOADING DATA
MODEL
TRAINING


Testing New Optimizers

In [ ]:
# Make sure we see your project code
import sys, importlib
sys.path.append('./python')
sys.path.append('./apps')

import needle
import models
import simple_ml

# Reload in case you edited optimizers / models / training code
importlib.reload(needle)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

# Pick device (cuda if available in your Needle backend)
try:
    device = ndl.cuda()
except Exception:
    device = ndl.cpu()
print("Using device:", device)

# CIFAR-10 train loader (same as before)
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(
    dataset=dataset,
    batch_size=128,
    shuffle=True,
)

# If you have a separate test set, you can also build a test loader here.
# For now, I'll just reuse `dataloader` for evaluate_cifar10 like your original code.


# List of optimizers to compare: (name, class, lr, weight_decay)
optim_experiments = [
    ("Adam", ndl.optim.Adam, 1e-3, 0.001),
    # ("SGD",  ndl.optim.SGD,  1e-1, 1e-4),  # tweak lr/wd as you like
    ("Muon", ndl.optim.Muon, 0.001, 0.001),
    ("SOAP", ndl.optim.SOAP, 0.001, 0.001),
]

results = {}

for name, opt_cls, lr, wd in optim_experiments:
    print(f"\n==============================")
    print(f"  Training with {name}")
    print(f"  lr={lr}, weight_decay={wd}")
    print(f"==============================")

    # New model for each optimizer so results are comparable
    model = ResNet9(device=device, dtype="float32")

    # Train
    train_acc, train_loss = train_cifar10(
        model,
        dataloader,
        n_epochs=30,            # match your Needle experiments
        optimizer=opt_cls,      # <- this is where your new optimizer is used
        lr=lr,
        weight_decay=wd,
    )

    # Evaluate (using same loader here; swap to test loader if you have one)
    eval_acc, eval_loss = evaluate_cifar10(model, dataloader)

    print(f"[{name}] final train_acc={train_acc:.4f}, train_loss={train_loss:.4f}")
    print(f"[{name}] final eval_acc ={eval_acc:.4f}, eval_loss ={eval_loss:.4f}")

    results[name] = {
        "train_acc": train_acc,
        "train_loss": train_loss,
        "eval_acc": eval_acc,
        "eval_loss": eval_loss,
    }

print("\nSummary:")
for name, stats in results.items():
    print(
        f"{name:5s} | "
        f"train_acc={stats['train_acc']:.4f}, train_loss={stats['train_loss']:.4f} | "
        f"eval_acc={stats['eval_acc']:.4f}, eval_loss={stats['eval_loss']:.4f}"
    )


Using device: cuda()

  Training with Adam
  lr=0.001, weight_decay=0.001
Epoch 00 | train_acc=0.3897, train_loss=1.6995 | test_acc=0.4813, test_loss=1.4310
Epoch 01 | train_acc=0.4944, train_loss=1.4007 | test_acc=0.5187, test_loss=1.3237
Epoch 02 | train_acc=0.5419, train_loss=1.2719 | test_acc=0.5621, test_loss=1.2185
Epoch 03 | train_acc=0.5789, train_loss=1.1803 | test_acc=0.5850, test_loss=1.1538
Epoch 04 | train_acc=0.6055, train_loss=1.1068 | test_acc=0.5688, test_loss=1.2161
Epoch 05 | train_acc=0.6310, train_loss=1.0423 | test_acc=0.5762, test_loss=1.1954
Epoch 06 | train_acc=0.6495, train_loss=0.9893 | test_acc=0.6045, test_loss=1.1162
Epoch 07 | train_acc=0.6674, train_loss=0.9402 | test_acc=0.6288, test_loss=1.0392
Epoch 08 | train_acc=0.6843, train_loss=0.8959 | test_acc=0.6221, test_loss=1.0627
Epoch 09 | train_acc=0.6967, train_loss=0.8578 | test_acc=0.5863, test_loss=1.1994
Epoch 10 | train_acc=0.7087, train_loss=0.8245 | test_acc=0.6325, test_loss=1.0577
Epoch 11 | tr

In [ ]:
# Make sure we see your project code
import sys, importlib
sys.path.append('./python')
sys.path.append('./apps')

import needle
import models
import simple_ml

# Reload in case you edited optimizers / models / training code
importlib.reload(needle)
importlib.reload(needle.optim)
importlib.reload(needle.ops.ops_mathematic)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

# Pick device (cuda if available in your Needle backend)
try:
    device = ndl.cuda()
except Exception:
    device = ndl.cpu()
print("Using device:", device)

# CIFAR-10 train loader (same as before)
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(
    dataset=dataset,
    batch_size=128,
    shuffle=True,
)

# If you have a separate test set, you can also build a test loader here.
# For now, I'll just reuse `dataloader` for evaluate_cifar10 like your original code.


# List of optimizers to compare: (name, class, lr, weight_decay)
optim_experiments = [
    # ("Adam", ndl.optim.Adam, 1e-3, 0.001),
    # ("SGD",  ndl.optim.SGD,  1e-1, 1e-4),  # tweak lr/wd as you like
    ("Muon", ndl.optim.Muon, 0.02, 0.001),
    ("SOAP", ndl.optim.SOAP, 0.001, 0.001),
]

results = {}

for name, opt_cls, lr, wd in optim_experiments:
    print(f"\n==============================")
    print(f"  Training with {name}")
    print(f"  lr={lr}, weight_decay={wd}")
    print(f"==============================")

    # New model for each optimizer so results are comparable
    model = ResNet9(device=device, dtype="float32")

    # Train
    train_acc, train_loss = train_cifar10(
        model,
        dataloader,
        n_epochs=30,            # match your Needle experiments
        optimizer=opt_cls,      # <- this is where your new optimizer is used
        lr=lr,
        weight_decay=wd,
    )

    # Evaluate (using same loader here; swap to test loader if you have one)
    eval_acc, eval_loss = evaluate_cifar10(model, dataloader)

    print(f"[{name}] final train_acc={train_acc:.4f}, train_loss={train_loss:.4f}")
    print(f"[{name}] final eval_acc ={eval_acc:.4f}, eval_loss ={eval_loss:.4f}")

    results[name] = {
        "train_acc": train_acc,
        "train_loss": train_loss,
        "eval_acc": eval_acc,
        "eval_loss": eval_loss,
    }

print("\nSummary:")
for name, stats in results.items():
    print(
        f"{name:5s} | "
        f"train_acc={stats['train_acc']:.4f}, train_loss={stats['train_loss']:.4f} | "
        f"eval_acc={stats['eval_acc']:.4f}, eval_loss={stats['eval_loss']:.4f}"
    )


Using device: cuda()

  Training with Muon
  lr=0.02, weight_decay=0.001
Epoch 00 | train_acc=0.3679, train_loss=1.7715 | test_acc=0.4688, test_loss=1.4596
Epoch 01 | train_acc=0.4932, train_loss=1.4035 | test_acc=0.5340, test_loss=1.2840
Epoch 02 | train_acc=0.5455, train_loss=1.2687 | test_acc=0.5586, test_loss=1.2362
Epoch 03 | train_acc=0.5804, train_loss=1.1765 | test_acc=0.5708, test_loss=1.2031
Epoch 04 | train_acc=0.6095, train_loss=1.0993 | test_acc=0.5995, test_loss=1.1116
Epoch 05 | train_acc=0.6319, train_loss=1.0358 | test_acc=0.5971, test_loss=1.1211
Epoch 06 | train_acc=0.6511, train_loss=0.9840 | test_acc=0.5861, test_loss=1.2043
Epoch 07 | train_acc=0.6691, train_loss=0.9363 | test_acc=0.5598, test_loss=1.3884
Epoch 08 | train_acc=0.6837, train_loss=0.8945 | test_acc=0.6019, test_loss=1.2078
Epoch 09 | train_acc=0.7001, train_loss=0.8562 | test_acc=0.6134, test_loss=1.1743
Epoch 10 | train_acc=0.7121, train_loss=0.8136 | test_acc=0.6588, test_loss=0.9986
Epoch 11 | tra

TypeError: Tensor.sum() got an unexpected keyword argument 'axis'

Using Pytorch

Using pytorch resnet 9

In [ ]:
# ================================
# 0. Setup & imports
# ================================
!pip install -q torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# 1. CIFAR-10 data loaders
# ================================
transform = T.Compose([
    T.ToTensor(),                     # [0, 1]
    T.Normalize((0.5, 0.5, 0.5),      # mean
                (0.5, 0.5, 0.5)),     # std
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2)

# ================================
# 2. ResNet9 definition (PyTorch)
#    matches your Needle version:
#    - ConvBN(3,16,7,4)
#    - ConvBN(16,32,3,2)
#    - ConvBN(32,32,3,1) x2 + residual
#    - ConvBN(32,64,3,2)
#    - ConvBN(64,128,3,2)
#    - ConvBN(128,128,3,1) x2 + residual
#    - Flatten, Linear(128,128), ReLU, Linear(128,10)
# ================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=k, stride=s,
                      padding=k // 2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNet9(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Layer 1-8 conv blocks
        self.conv1 = ConvBNReLU(3,   16, 7, 4)
        self.conv2 = ConvBNReLU(16,  32, 3, 2)
        self.conv3 = ConvBNReLU(32,  32, 3, 1)
        self.conv4 = ConvBNReLU(32,  32, 3, 1)

        self.conv5 = ConvBNReLU(32,  64, 3, 2)
        self.conv6 = ConvBNReLU(64, 128, 3, 2)
        self.conv7 = ConvBNReLU(128,128, 3, 1)
        self.conv8 = ConvBNReLU(128,128, 3, 1)

        # Fully connected part
        # After conv6 on CIFAR-10 with these strides, spatial size is 1x1,
        # so feature dim = 128.
        self.fc1 = nn.Linear(128, 128)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input: (N, 3, 32, 32)
        x = self.conv1(x)
        out2 = self.conv2(x)

        x = self.conv3(out2)
        x = self.conv4(x)

        # Residual from layer 2
        x = x + out2

        x = self.conv5(x)
        out6 = self.conv6(x)

        x = self.conv7(out6)
        x = self.conv8(x)

        # Flatten both x and out6, add residual
        x_flat   = x.view(x.size(0), -1)      # (N, 128 * 1 * 1) = (N, 128)
        out6_flat = out6.view(out6.size(0), -1)
        x = x_flat + out6_flat

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = ResNet9().to(device)
# print(model)

# ================================
# 3. Loss & optimizer
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

# ================================
# 4. Train / eval helpers
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


# ================================
# 5. Main training loop
# ================================
num_epochs = 30   # set to 20/30 to mirror your Needle run

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, test_loss   = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Using device: cuda
Epoch 01 | train_acc=0.4313, train_loss=1.5592 | test_acc=0.5117, test_loss=1.3663
Epoch 02 | train_acc=0.5338, train_loss=1.2988 | test_acc=0.5470, test_loss=1.2538
Epoch 03 | train_acc=0.5810, train_loss=1.1713 | test_acc=0.5694, test_loss=1.2014
Epoch 04 | train_acc=0.6115, train_loss=1.0887 | test_acc=0.5870, test_loss=1.1658
Epoch 05 | train_acc=0.6397, train_loss=1.0145 | test_acc=0.5956, test_loss=1.1363
Epoch 06 | train_acc=0.6588, train_loss=0.9566 | test_acc=0.6089, test_loss=1.1100
Epoch 07 | train_acc=0.6796, train_loss=0.9021 | test_acc=0.6172, test_loss=1.0962
Epoch 08 | train_acc=0.6991, train_loss=0.8561 | test_acc=0.6242, test_loss=1.0664
Epoch 09 | train_acc=0.7134, train_loss=0.8164 | test_acc=0.6297, test_loss=1.0754
Epoch 10 | train_acc=0.7240, train_loss=0.7830 | test_acc=0.6269, test_loss=1.0870
Epoch 11 | train_acc=0.7369, train_loss=0.7489 | test_acc=0.6323, test_loss=1.0624
Epoch 12 | train_acc=0.7484, train_loss=0.7169 | test_acc=0.6301, te

Using resnet18

In [ ]:
# ==== 0. (Optional) install, usually already in Colab ====
# !pip install -q torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==== 1. Data: CIFAR-10 loaders ====
transform = T.Compose([
    T.ToTensor(),                     # [0,1]
    T.Normalize((0.5, 0.5, 0.5),      # mean
                (0.5, 0.5, 0.5)),     # std
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2)

# ==== 2. Model: simple ResNet (torchvision) ====
from torchvision.models import resnet18

model = resnet18(weights=None, num_classes=10)  # plain ResNet-18 for CIFAR-10
model = model.to(device)

# ==== 3. Loss & optimizer ====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

# ==== 4. Train & eval loops ====
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


# ==== 5. Main training loop ====
num_epochs = 30   # change to 20 / 30 to match your needle run

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, test_loss   = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Using device: cuda
Epoch 01 | train_acc=0.5058, train_loss=1.3755 | test_acc=0.5900, test_loss=1.1780
Epoch 02 | train_acc=0.6531, train_loss=0.9868 | test_acc=0.6603, test_loss=0.9724
Epoch 03 | train_acc=0.7059, train_loss=0.8463 | test_acc=0.6767, test_loss=0.9235
Epoch 04 | train_acc=0.7385, train_loss=0.7562 | test_acc=0.6981, test_loss=0.8639
Epoch 05 | train_acc=0.7621, train_loss=0.6891 | test_acc=0.7124, test_loss=0.8481
Epoch 06 | train_acc=0.7820, train_loss=0.6313 | test_acc=0.7288, test_loss=0.7992
Epoch 07 | train_acc=0.8022, train_loss=0.5766 | test_acc=0.7468, test_loss=0.7525
Epoch 08 | train_acc=0.8195, train_loss=0.5258 | test_acc=0.7537, test_loss=0.7352
Epoch 09 | train_acc=0.8366, train_loss=0.4781 | test_acc=0.7413, test_loss=0.7900
Epoch 10 | train_acc=0.8466, train_loss=0.4493 | test_acc=0.7523, test_loss=0.7462
Epoch 11 | train_acc=0.8630, train_loss=0.4031 | test_acc=0.7643, test_loss=0.7471
Epoch 12 | train_acc=0.8755, train_loss=0.3662 | test_acc=0.7527, te